In [1]:
import os
import pandas as pd
import requests

In [4]:
# 1. Configuration parameters
# Lat/Lon for center of Narail District, Bangladesh
LATITUDE = 23.1667
LONGITUDE = 89.5000
# Target Time Window: 2-Year Historical Frame (Jan 2023 - Dec 2024)
START_DATE = "20230101"
END_DATE = "20241231"
# Parameters mapping to system attributes:
# ALLSKY_SFC_SW_DWN = Global Horizontal Irradiance (GHI) - Core input for Solar PV prediction
# CLRSKY_SFC_SW_DWN = Clear Sky Solar Irradiance - Essential for computing cloud attenuation
# T2M               = Ambient Temperature at 2m - Governing factor for PV thermal efficiency and digester heating
# RH2M              = Relative Humidity at 2m
# WS10M             = Wind Speed at 10m - Controls ambient convective cooling
PARAMS = ["ALLSKY_SFC_SW_DWN", "CLRSKY_SFC_SW_DWN", "T2M", "RH2M", "WS10M"]

In [ ]:
def fetch_nasa_power_data(lat, lon, start, end, parameters):
    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters={','.join(parameters)}&"
        f"community=RE&"  # Renewable Energy community profile
        f"longitude={lon}&"
        f"latitude={lat}&"
        f"start={start}&"
        f"end={end}&"
        f"format=JSON"
    )

    print(f"Connecting to NASA API for Coordinates: Lat {lat}, Lon {lon}...")
    response = requests.get(url, timeout=30)

    if response.status_code == 200:
        json_data = response.json()
        # Parse parameter data dictionary
        raw_records = json_data["properties"]["parameter"]

        # Convert nested dictionaries into standard DataFrame
        df = pd.DataFrame(raw_records)
        df.index = pd.to_datetime(df.index, format="%Y%m%d")
        df.index.name = "Date"

        # Relabel parameters into highly descriptive engineering headers
        df = df.rename(
            columns={
                "ALLSKY_SFC_SW_DWN": "GHI_Satellite",
                "CLRSKY_SFC_SW_DWN": "ClearSky_GHI",
                "T2M": "Ambient_Temp",
                "RH2M": "Humidity",
                "WS10M": "Wind_Speed",
            }
        )
        return df
    else:
        raise Exception(
            f"API Connection Failed. Status: {response.status_code}, Msg: {response.text}"
        )


In [6]:
try:
    master_weather_df = fetch_nasa_power_data(
        LATITUDE, LONGITUDE, START_DATE, END_DATE, PARAMS
    )
    master_weather_df.to_csv("data/downloaded_weather_base.csv")
    print("\nInitialization Complete!")
    print(f"Successfully saved {len(master_weather_df)} records.")
    print(master_weather_df.head())
except Exception as e:
    print(f"Execution Error: {e}")

Connecting to NASA API for Coordinates: Lat 23.1667, Lon 89.5...

Initialization Complete!
Successfully saved 731 records.
            GHI_Satellite  ClearSky_GHI  Ambient_Temp  Humidity  Wind_Speed
Date                                                                       
2023-01-01         3.0077        3.3367         17.16     76.93        2.38
2023-01-02         3.2676        3.3418         17.21     76.45        2.59
2023-01-03         1.5967        3.6482         16.71     76.99        2.79
2023-01-04         1.1642        3.5674         15.45     77.22        3.06
2023-01-05         2.7022        3.5978         15.23     75.65        3.32
